In [1]:
# ==========================================================
# ACTIVIDAD 12. Filtros revisados y diagnóstico de saltos
# Sin filtro de precio (el precio de Yahoo está ajustado por
# contrasplits futuros). Liquidez en dólares, validez y una
# clase por empresa. Después, diagnóstico de saltos extremos.
# ==========================================================
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

VOLUMEN_DOLARES_MIN = 5_000_000   # propuesta: mediana diaria en el entrenamiento
DIAS_SIN_VOLUMEN_MAX = 0.01
SALTO = 0.4                       # |rendimiento logarítmico| > 0,4 (≈ +49 % o -33 %)
REVERSION = 0.1                   # el día siguiente deshace el salto casi por completo

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Métricas de la Actividad 11
# ----------------------------------------------------------
# na_values=[""] : solo las celdas vacías son nulos; un ticker "NA" se conserva
metricas = pd.read_csv(CARPETA / "metricas_entrenamiento.csv",
                       dtype={"ticker": str, "cik": str},
                       keep_default_na=False, na_values=[""]).set_index("ticker")
print("Tickers:", len(metricas))

valido = (metricas["n_no_positivo"] == 0) & (metricas["cociente_min"] > 0)
negocia = metricas["pct_sin_volumen"] <= DIAS_SIN_VOLUMEN_MAX

def una_por_empresa(seleccion):
    """De cada empresa (CIK) conserva la clase más líquida."""
    return (metricas[seleccion].sort_values("volumen_dolares_mediano", ascending=False)
            .drop_duplicates("cik"))

# ----------------------------------------------------------
# 2. Sensibilidad al umbral de liquidez (sin filtro de precio)
# ----------------------------------------------------------
print("\n=== EMPRESAS ELEGIBLES SEGÚN EL VOLUMEN MÍNIMO ===")
for umbral in [1_000_000, 2_000_000, 5_000_000, 10_000_000, 20_000_000]:
    n = len(una_por_empresa(valido & negocia & (metricas["volumen_dolares_mediano"] >= umbral)))
    print(f"Volumen mediano >= {umbral / 1e6:>4.0f} M$: {n:5d} empresas")

# ----------------------------------------------------------
# 3. Embudo con el umbral propuesto
# ----------------------------------------------------------
liquido = metricas["volumen_dolares_mediano"] >= VOLUMEN_DOLARES_MIN
print("\n=== EMBUDO (tickers | empresas) ===")
acumulado = pd.Series(True, index=metricas.index)
for nombre, condicion in [("1. Universo", acumulado),
                          ("2. Datos válidos", valido),
                          (f"3. Volumen mediano >= {VOLUMEN_DOLARES_MIN / 1e6:.0f} M$", liquido),
                          (f"4. Sesiones sin volumen <= {DIAS_SIN_VOLUMEN_MAX:.0%}", negocia)]:
    acumulado = acumulado & condicion
    print(f"{nombre:35s} {acumulado.sum():5d} | {metricas.loc[acumulado, 'cik'].nunique():5d}")
elegibles = una_por_empresa(acumulado)
print(f"{'5. Una clase por empresa':35s} {len(elegibles):5d} | {elegibles['cik'].nunique():5d}")

# ----------------------------------------------------------
# 4. Saltos extremos en TODO el periodo (diagnóstico de calidad)
# ----------------------------------------------------------
precios = pd.read_parquet(CARPETA_DATOS / "precios_universo.parquet",
                          columns=["ticker", "date", "adj_close"])
precios = precios[precios["ticker"].isin(elegibles.index)].copy()
precios["ticker"] = precios["ticker"].astype(str)
precios["r"] = np.log(precios["adj_close"]).groupby(precios["ticker"]).diff()
precios["r_siguiente"] = precios.groupby("ticker")["r"].shift(-1)

# Salto revertido: un movimiento grande que el día siguiente deshace casi por completo.
# Es la huella típica de un dato erróneo aislado (por ejemplo, un contrasplit mal ajustado).
precios["salto_revertido"] = ((precios["r"].abs() > SALTO)
                              & (precios["r"] * precios["r_siguiente"] < 0)
                              & ((precios["r"] + precios["r_siguiente"]).abs() < REVERSION))
precios["salto_extremo"] = precios["r"].abs() > np.log(2)     # se duplica o se reduce a la mitad

saltos = precios.groupby("ticker").agg(
    max_r_abs=("r", lambda r: r.abs().max()),
    saltos_extremos=("salto_extremo", "sum"),
    saltos_revertidos=("salto_revertido", "sum"),
)
print("\n=== SALTOS EN LAS EMPRESAS ELEGIBLES (todo el periodo) ===")
print("Con algún salto extremo (x2 o /2 en un día):", (saltos["saltos_extremos"] > 0).sum())
print("Con algún salto revertido (posible error de datos):", (saltos["saltos_revertidos"] > 0).sum())
print("\nLos 20 mayores saltos (r logarítmico):")
print(saltos.sort_values("max_r_abs", ascending=False).head(20))

print("\nEjemplo: los días alrededor del mayor salto")
peor = saltos["max_r_abs"].idxmax()
fila = precios.loc[precios["ticker"] == peor, "r"].abs().idxmax()
print(precios.loc[fila - 3: fila + 3, ["ticker", "date", "adj_close", "r"]].to_string(index=False))

# ----------------------------------------------------------
# 5. Estratos de liquidez y volatilidad (solo entrenamiento)
# ----------------------------------------------------------
elegibles = elegibles.join(saltos)
elegibles["liquidez"] = pd.qcut(elegibles["volumen_dolares_mediano"], 3, labels=["baja", "media", "alta"])
elegibles["volatilidad"] = pd.qcut(elegibles["volatilidad_anual"], 3, labels=["baja", "media", "alta"])
print("\n=== ESTRATOS: liquidez (filas) x volatilidad (columnas) ===")
print(pd.crosstab(elegibles["liquidez"], elegibles["volatilidad"], margins=True, margins_name="Total"))
print("\nLímites de los terciles:")
print("Volumen (M$):", (elegibles["volumen_dolares_mediano"].quantile([1 / 3, 2 / 3]) / 1e6).round(1).tolist())
print("Volatilidad anual:", elegibles["volatilidad_anual"].quantile([1 / 3, 2 / 3]).round(3).tolist())
print("\nPor división SIC:")
print(elegibles["division_sic"].value_counts())

# ----------------------------------------------------------
# 6. Guardar
# ----------------------------------------------------------
elegibles.to_csv(CARPETA / "elegibles_muestra.csv", index_label="ticker")
print("\nGuardado: elegibles_muestra.csv -", len(elegibles), "empresas")

Mounted at /content/drive
Tickers: 2823

=== EMPRESAS ELEGIBLES SEGÚN EL VOLUMEN MÍNIMO ===
Volumen mediano >=    1 M$:  2043 empresas
Volumen mediano >=    2 M$:  1874 empresas
Volumen mediano >=    5 M$:  1593 empresas
Volumen mediano >=   10 M$:  1353 empresas
Volumen mediano >=   20 M$:  1048 empresas

=== EMBUDO (tickers | empresas) ===
1. Universo                          2823 |  2789
2. Datos válidos                     2820 |  2786
3. Volumen mediano >= 5 M$           1610 |  1601
4. Sesiones sin volumen <= 1%        1602 |  1593
5. Una clase por empresa             1593 |  1593

=== SALTOS EN LAS EMPRESAS ELEGIBLES (todo el periodo) ===
Con algún salto extremo (x2 o /2 en un día): 77
Con algún salto revertido (posible error de datos): 14

Los 20 mayores saltos (r logarítmico):
        max_r_abs  saltos_extremos  saltos_revertidos
ticker                                               
CHRD        5.554                6                  0
IRLA        2.466                4       